# A satellite orbiting the Earth, orbiting the Sun

A satellite going round the Earth is *also* being carried round the Sun at about 30 km/s. Seen from the Sun, its path is the Earth's orbit with a wobble added on top, which is the same idea as an ancient **epicycle**.

The notebook has two parts:

1. **A real simulation.** It uses Newtonian gravity from the Sun and the Earth, with a symplectic (leapfrog) integrator so the orbits do not drift.
2. **An exaggerated model.** It shows how the shape of the path depends on one number: how fast the satellite moves round the Earth compared with how fast the Earth moves round the Sun.

Run the cells in order. Everything uses only `numpy` and `matplotlib`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

G       = 6.67430e-11     # gravitational constant (m^3 kg^-1 s^-2)
M_SUN   = 1.989e30        # kg
M_EARTH = 5.972e24        # kg
AU      = 1.496e11        # Earth-Sun distance (m)
DAY     = 86400.0         # s
YEAR    = 365.25 * DAY    # s

def simulate(r_sat=3.844e8, days=365.25, retrograde=False, steps_per_orbit=400):
    """Sun, Earth and a satellite in the plane, integrated with velocity-Verlet (leapfrog).

    r_sat      : starting distance of the satellite from the Earth's centre (m).
                 The default is the Moon's distance. Keep it below about 7e8 m,
                 or the Sun pulls the satellite away from the Earth.
    retrograde : True to send the satellite round the Earth the opposite way.
    Returns time (s) and the x, y positions of Earth and satellite relative to the Sun.
    The Sun is fixed at the origin. The satellite's mass is negligible.
    """
    v_earth = np.sqrt(G * M_SUN / AU)
    v_rel   = np.sqrt(G * M_EARTH / r_sat) * (-1 if retrograde else 1)
    T_sat   = 2 * np.pi * np.sqrt(r_sat**3 / (G * M_EARTH))
    dt      = min(T_sat / steps_per_orbit, 3600.0)
    n       = int(days * DAY / dt)

    e = np.array([AU, 0.0]);          ve = np.array([0.0, v_earth])
    s = np.array([AU + r_sat, 0.0]);  vs = np.array([0.0, v_earth + v_rel])

    def acc(e, s):
        a_e = -G * M_SUN * e / np.linalg.norm(e)**3
        d = s - e
        a_s = (-G * M_SUN * s / np.linalg.norm(s)**3
               - G * M_EARTH * d / np.linalg.norm(d)**3)
        return a_e, a_s

    out = np.empty((n + 1, 5)); out[0] = (0, *e, *s)
    a_e, a_s = acc(e, s)
    for i in range(1, n + 1):
        ve += 0.5 * dt * a_e;  vs += 0.5 * dt * a_s
        e  += dt * ve;         s  += dt * vs
        a_e, a_s = acc(e, s)
        ve += 0.5 * dt * a_e;  vs += 0.5 * dt * a_s
        out[i] = (i * dt, *e, *s)
    t, xe, ye, xs, ys = out.T
    return t, xe, ye, xs, ys

def plot_orbits(t, xe, ye, xs, ys, link_every_days=6, zoom_days=120, magnify=20):
    fig, axs = plt.subplots(1, 3, figsize=(16, 5.6), constrained_layout=True)
    # (a) true scale
    ax = axs[0]
    ax.plot(xe / AU, ye / AU, 'C0', lw=1, label='Earth')
    ax.plot(xs / AU, ys / AU, 'C3', lw=1, label='Satellite')
    ax.plot(0, 0, 'o', color='gold', ms=14, mec='darkorange', label='Sun')
    ax.set_aspect('equal'); ax.set_xlabel('x (AU)'); ax.set_ylabel('y (AU)')
    ax.set_title('(a) Seen from the Sun, true scale\nthe two paths are indistinguishable'); ax.legend(loc='lower left', fontsize=8)
    # (b) relative to Earth
    ax = axs[1]
    dx, dy = (xs - xe) / 1e6, (ys - ye) / 1e6
    sc = ax.scatter(dx, dy, c=t / DAY, s=2, cmap='viridis')
    ax.plot(0, 0, 'o', color='C0', ms=9, label='Earth')
    ax.set_aspect('equal'); ax.set_xlabel('x relative to Earth (thousand km)'); ax.set_ylabel('y relative to Earth (thousand km)')
    ax.set_title('(b) Seen from the Earth\nthe satellite goes round the Earth'); plt.colorbar(sc, ax=ax, label='day of year', fraction=.046); ax.legend(loc='upper right', fontsize=8)
    # (c) zoom on the epicycle, with the satellite's distance from the Earth magnified
    ax = axs[2]
    m = t <= zoom_days * DAY
    xm = xe + magnify * (xs - xe); ym = ye + magnify * (ys - ye)
    ax.plot(xe[m] / 1e9, ye[m] / 1e9, 'C0', lw=2, label='Earth (centre of the epicycle)')
    ax.plot(xm[m] / 1e9, ym[m] / 1e9, 'C3', lw=1.2, label=f'Satellite (offset from Earth x{magnify})')
    step = max(1, int(round(link_every_days * DAY / (t[1] - t[0]))))
    idx = np.arange(0, m.sum(), step)
    for i in idx:
        ax.plot([xe[i] / 1e9, xm[i] / 1e9], [ye[i] / 1e9, ym[i] / 1e9], color='gray', lw=.6, alpha=.7)
    ax.plot(xe[idx] / 1e9, ye[idx] / 1e9, 'o', color='C0', ms=3)
    ax.set_aspect('equal'); ax.set_xlabel('x (million km)'); ax.set_ylabel('y (million km)')
    ax.set_title(f'(c) First {zoom_days} days, enlarged\nsatellite offset magnified x{magnify} so the wobble shows'); ax.legend(loc='upper left', fontsize=8)
    plt.show()


In [ ]:
# The satellite is as far from the Earth as the Moon.
# Try changing r_sat (in metres) and retrograde=True. Keep r_sat below about 7e8.
t, xe, ye, xs, ys = simulate(r_sat=3.844e8, days=365.25)
plot_orbits(t, xe, ye, xs, ys)

## Why the path never makes a loop

In panel (c) the satellite's distance from the Earth is magnified by 20, and its path is still only a gentle wave around the Earth's track. It never loops back on itself. The reason is a comparison of speeds:

- The Earth moves round the Sun at about **29.8 km/s**.
- A satellite at the Moon's distance moves round the Earth at only about **1.0 km/s**.

A loop, which is what an ancient epicycle looks like, needs the satellite's speed round the Earth to be **larger** than the Earth's speed round the Sun. Real satellites never manage that, because gravity does not allow it at distances where the Earth still holds them. The real Moon's ratio is only 1.0 / 29.8 = 0.034. The next cell shows what the shape would be for other ratios. It is a scaled-down model and **not** to scale: the satellite orbit is drawn at 12% of the Earth's orbital radius, so the satellite orbits many more times per year than a real one would at that speed ratio.

In [ ]:
def epicycle(ratio, r_over_R=0.12, years=1.0, n=4000):
    """Path of a satellite circling a body that circles the Sun (both circular, same direction).
    ratio = (satellite speed round the Earth) / (Earth's speed round the Sun)."""
    R = 1.0
    w_earth = 2 * np.pi            # one orbit per year
    r = r_over_R * R
    w_sat = ratio * (R * w_earth) / r   # speed = r * w_sat
    t = np.linspace(0, years, n)
    xe, ye = R * np.cos(w_earth * t), R * np.sin(w_earth * t)
    xs, ys = xe + r * np.cos(w_sat * t), ye + r * np.sin(w_sat * t)
    return t, xe, ye, xs, ys, w_sat / w_earth

cases = [(0.3, 'Much slower: ratio 0.3 (gentle wave)'),
         (0.7, 'Slower: ratio 0.7 (bulges)'),
         (1.0, 'Equal speeds: ratio 1 (cusps)'),
         (3.0, 'Faster: ratio 3 (loops)')]

fig, axs = plt.subplots(1, 4, figsize=(18, 5.0), constrained_layout=True)
for ax, (ratio, title) in zip(axs, cases):
    t, xe, ye, xs, ys, orbits = epicycle(ratio, years=1.0, n=6000)
    ax.plot(xe, ye, color='C0', lw=1, alpha=.6, label='Earth')
    ax.plot(xs, ys, color='C3', lw=.8, label='Satellite')
    ax.plot(0, 0, 'o', color='gold', mec='darkorange', ms=10)
    ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f'{title}\n({orbits:.1f} satellite orbits per year)', fontsize=10)
axs[0].legend(loc='lower left', fontsize=8)
plt.show()

## Things to try

- In the first simulation, set `retrograde=True`. The satellite then orbits the Earth the opposite way round. Does the shape in panel (c) change? Look at the extremes of the distance in panel (b) too.
- Change `r_sat` to `1e8` (about a quarter of the Moon's distance). Is the epicycle in panel (c) larger or smaller? Does the satellite orbit the Earth more often?
- Push `r_sat` up towards `7e8` m and watch panel (b). The Sun distorts the orbit more and more. At `1e9` the satellite is pulled away from the Earth during the year.
- In `epicycle(...)`, change `r_over_R` to `0.3` or `0.05` and see how the loops grow or shrink.
- Which ratio of speeds first gives loops? Check it against what you see in the last panel.